# HW7 Q1: Pruning the 4-bit VGG16_quant
Start from the 4-bit QAT VGG16_quant of HW6 (the baseline), prune **every convolution layer** to **80% sparsity**, and retrain with the pruning masks kept so the pruned weights stay zero. Done twice from the same baseline:
- **unstructured**: `prune.l1_unstructured` zeroes the 80% of weights with the smallest magnitude in each layer
- **structured**: `prune.ln_structured(dim=0, n=1)` removes the 80% of output filters with the smallest L1 norm in each layer

## Setup

In [ ]:
import argparse
import os
import time
import shutil

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.backends.cudnn as cudnn

    

import torchvision
import torchvision.transforms as transforms

import sys
sys.path.insert(0, os.path.abspath('..'))   # this notebook is in software/hw/, the models/ folder is in software/
from models import *   # bring everything in the folder models

global best_prec
use_gpu = torch.cuda.is_available()
device = 'cuda' if use_gpu else 'cpu'
print('device:', device)
print('=> Building model...')
    
    
batch_size = 128

model_name = "VGG16_quant"
model = VGG16_quant()   # 4-bit QuantConv2d in every conv layer (models/vgg_quant.py)

        

normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])


train_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=True,
    download=True,
    transform=transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        normalize,
    ]))
trainloader = torch.utils.data.DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)


test_dataset = torchvision.datasets.CIFAR10(
    root='../data',
    train=False,
    download=True,
    transform=transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ]))

testloader = torch.utils.data.DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)


print_freq = 100 # every 100 batches, accuracy printed. Here, each batch includes "batch_size" data points
# CIFAR10 has 50,000 training data, and 10,000 validation data.

def train(trainloader, model, criterion, optimizer, epoch):
    batch_time = AverageMeter()   ## at the begining of each epoch, this should be reset
    data_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    model.train()

    end = time.time()  # measure current time
    
    for i, (input, target) in enumerate(trainloader):
        # measure data loading time
        data_time.update(time.time() - end)  # data loading time

        input, target = input.to(device), target.to(device)

        # compute output
        output = model(input)
        loss = criterion(output, target)

        # measure accuracy and record loss
        prec = accuracy(output, target)[0]
        losses.update(loss.item(), input.size(0))
        top1.update(prec.item(), input.size(0))

        # compute gradient and do SGD step
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        # measure elapsed time
        batch_time.update(time.time() - end) # time spent to process one batch
        end = time.time()


        if i % print_freq == 0:
            print('Epoch: [{0}][{1}/{2}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Data {data_time.val:.3f} ({data_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   epoch, i, len(trainloader), batch_time=batch_time,
                   data_time=data_time, loss=losses, top1=top1))

            

def validate(val_loader, model, criterion ):
    batch_time = AverageMeter()
    losses = AverageMeter()
    top1 = AverageMeter()

    # switch to evaluate mode
    model.eval()

    end = time.time()
    with torch.no_grad():
        for i, (input, target) in enumerate(val_loader):
         
            input, target = input.to(device), target.to(device)

            # compute output
            output = model(input)
            loss = criterion(output, target)

            # measure accuracy and record loss
            prec = accuracy(output, target)[0]
            losses.update(loss.item(), input.size(0))
            top1.update(prec.item(), input.size(0))

            # measure elapsed time
            batch_time.update(time.time() - end)
            end = time.time()

            if i % print_freq == 0:  # This line shows how frequently print out the status. e.g., i%5 => every 5 batch, prints out
                print('Test: [{0}/{1}]\t'
                  'Time {batch_time.val:.3f} ({batch_time.avg:.3f})\t'
                  'Loss {loss.val:.4f} ({loss.avg:.4f})\t'
                  'Prec {top1.val:.3f}% ({top1.avg:.3f}%)'.format(
                   i, len(val_loader), batch_time=batch_time, loss=losses,
                   top1=top1))

    print(' * Prec {top1.avg:.3f}% '.format(top1=top1))
    return top1.avg


def accuracy(output, target, topk=(1,5)):
    """Computes the precision@k for the specified values of k"""
    maxk = max(topk) # 5
    batch_size = target.size(0) # 128

    _, pred = output.topk(maxk, 1, True, True) # topk(k, dim=None, largest=True, sorted=True)
                                    # will output (max value, its index)
    pred = pred.t()               # transpose
    correct = pred.eq(target.view(1, -1).expand_as(pred))   # "-1": calculate automatically

    res = []
    for k in topk: # 1, 5
        correct_k = correct[:k].reshape(-1).float().sum(0)  # view(-1): make a flattened 1D tensor
        res.append(correct_k.mul_(100.0 / batch_size))   # correct: size of [maxk, batch_size]
    return res


class AverageMeter(object):
    """Computes and stores the average and current value"""
    def __init__(self):
        self.reset()

    def reset(self):
        self.val = 0
        self.avg = 0
        self.sum = 0
        self.count = 0

    def update(self, val, n=1):
        self.val = val
        self.sum += val * n    ## n is impact factor
        self.count += n
        self.avg = self.sum / self.count

        
def save_checkpoint(state, is_best, fdir):
    filepath = os.path.join(fdir, 'checkpoint.pth')
    torch.save(state, filepath)
    if is_best:
        shutil.copyfile(filepath, os.path.join(fdir, 'model_best.pth.tar'))


def adjust_learning_rate(optimizer, epoch):
    """For resnet, the lr starts from 0.1, and is divided by 10 at 80 and 120 epochs"""
    adjust_list = [150, 225]
    if epoch in adjust_list:
        for param_group in optimizer.param_groups:
            param_group['lr'] = param_group['lr'] * 0.1        

#model = nn.DataParallel(model).cuda()
#all_params = checkpoint['state_dict']
#model.load_state_dict(all_params, strict=False)
#criterion = nn.CrossEntropyLoss().cuda()
#validate(testloader, model, criterion)

In [ ]:
import torch.nn.utils.prune as prune

baseline_path = 'result/VGG16_quant/model_best.pth.tar'     # 4-bit QAT checkpoint from HW6
sparsity = 0.8
criterion = nn.CrossEntropyLoss().to(device)

def conv_layers(model):
    # every convolution layer (all of them are QuantConv2d in VGG16_quant)
    return [(i, m) for i, m in enumerate(model.features) if isinstance(m, nn.Conv2d)]

def load_baseline():
    model = VGG16_quant()
    model.load_state_dict(torch.load(baseline_path, map_location=device)['state_dict'])
    return model.to(device)

def apply_pruning(model, kind):
    for _, m in conv_layers(model):
        if kind == 'unstructured':
            prune.l1_unstructured(m, name='weight', amount=sparsity)
        else:   # structured: whole output filters (dim=0), ranked by L1 norm
            prune.ln_structured(m, name='weight', amount=sparsity, dim=0, n=1)
    return model

def load_pruned(kind):
    # a pruned checkpoint stores weight_orig and weight_mask, so prune the fresh model first, then load it
    model = apply_pruning(VGG16_quant(), kind)
    model.load_state_dict(torch.load('result/VGG16_quant_pruned_'+kind+'/model_best.pth.tar', map_location=device)['state_dict'])
    return model.to(device)

def sparsity_report(model):
    # sparsity measured on weight_mask (fraction of pruned weights), and on the weights themselves
    print('layer          shape              mask sparsity   weight == 0')
    pruned = total = 0
    for i, m in conv_layers(model):
        mask = m.weight_mask
        print('features[{:2d}]   {:18s} {:10.2%} {:12.2%}'.format(
            i, str(list(m.weight.shape)), (mask == 0).float().mean().item(), (m.weight == 0).float().mean().item()))
        pruned += (mask == 0).sum().item()
        total += mask.numel()
    print('all conv layers: {:.2%} of {} weights pruned'.format(pruned / total, total))

results = {}

## Baseline: 4-bit VGG16_quant from HW6

In [ ]:
model = load_baseline()
results['Baseline (4-bit QAT, no pruning)'] = validate(testloader, model, criterion)
print('baseline test accuracy: {:.2f}%'.format(results['Baseline (4-bit QAT, no pruning)']))

## Retraining function

In [ ]:
def retrain(model, kind, lr=0.02, epochs=40, weight_decay=5e-4):
    # Same QAT optimizer setup as HW6. The masks stay on: prune only re-parametrizes weight = weight_orig * weight_mask,
    # so the pruned weights stay zero (prune.remove is never called). weight_orig is trained, weight_q is not.
    alpha_params = [p for n, p in model.named_parameters() if 'alpha' in n]
    other_params = [p for n, p in model.named_parameters() if 'alpha' not in n and 'weight_q' not in n]
    optimizer = torch.optim.SGD([{'params': other_params, 'weight_decay': weight_decay},
                                 {'params': alpha_params, 'weight_decay': 0}], lr=lr, momentum=0.9)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    fdir = 'result/VGG16_quant_pruned_'+kind
    os.makedirs(fdir, exist_ok=True)
    best_prec = 0
    for epoch in range(epochs):
        train(trainloader, model, criterion, optimizer, epoch)
        prec = validate(testloader, model, criterion)
        is_best = prec > best_prec
        best_prec = max(prec, best_prec)
        print('best acc: {:1f}'.format(best_prec))
        save_checkpoint({'epoch': epoch + 1, 'state_dict': model.state_dict(), 'best_prec': best_prec,
                         'optimizer': optimizer.state_dict()}, is_best, fdir)
        scheduler.step()

# 1. Unstructured pruning (80%)

In [ ]:
model = apply_pruning(load_baseline(), 'unstructured')
print(list(dict(model.features[3].named_parameters()).keys()), '+ buffer', list(dict(model.features[3].named_buffers()).keys()))
sparsity_report(model)
results['Unstructured, right after pruning'] = validate(testloader, model, criterion)
print('unstructured pruning, before retraining: {:.2f}%'.format(results['Unstructured, right after pruning']))

### Retraining

In [ ]:
# Retrain the pruned model and save the best checkpoint to result/VGG16_quant_pruned_unstructured/.
torch.manual_seed(0)
model = apply_pruning(load_baseline(), 'unstructured')
retrain(model, 'unstructured')

### After retraining

In [ ]:
model = load_pruned('unstructured')
results['Unstructured, after retraining'] = validate(testloader, model, criterion)
print('unstructured pruning, after retraining: {:.2f}%'.format(results['Unstructured, after retraining']))
sparsity_report(model)     # masks kept: still 80% sparse
model_unstructured = model

# 2. Structured pruning (80% of the output filters)

In [ ]:
model = apply_pruning(load_baseline(), 'structured')
sparsity_report(model)
w = model.features[3].weight
print('features[3]: {} of {} output filters are all zero'.format(int((w.flatten(1).abs().sum(1) == 0).sum()), w.shape[0]))
results['Structured, right after pruning'] = validate(testloader, model, criterion)
print('structured pruning, before retraining: {:.2f}%'.format(results['Structured, right after pruning']))

### Retraining

In [ ]:
# Same settings as the unstructured run. Saves the best checkpoint to result/VGG16_quant_pruned_structured/.
torch.manual_seed(0)
model = apply_pruning(load_baseline(), 'structured')
retrain(model, 'structured')

### After retraining

In [ ]:
model = load_pruned('structured')
results['Structured, after retraining'] = validate(testloader, model, criterion)
print('structured pruning, after retraining: {:.2f}%'.format(results['Structured, after retraining']))
sparsity_report(model)

## Note: sparsity of the quantized weights
`QuantConv2d` normalizes the weights (subtracts the layer mean, divides by the standard deviation) before quantizing, so a pruned weight of 0 becomes −mean/std, which can round to a non-zero integer. `weight_q` and `weight_int` are therefore not guaranteed to keep the 80% sparsity, which is why sparsity is measured on `weight_mask` above. For `features[3]` of the retrained unstructured model:

In [ ]:
layer = model_unstructured.features[3]
model_unstructured.eval()
with torch.no_grad():
    model_unstructured(next(iter(testloader))[0][:1].to(device))    # forward() refreshes weight_q
w_int = (layer.weight_q / (layer.weight_quant.wgt_alpha / (2 ** (4 - 1) - 1))).round()
print('weight_mask sparsity: {:.2%}   weight == 0: {:.2%}   weight_int == 0: {:.2%}'.format(
    (layer.weight_mask == 0).float().mean().item(), (layer.weight == 0).float().mean().item(), (w_int == 0).float().mean().item()))

## Results

In [ ]:
print('{:40s} {:>12s}'.format('Model', 'Test acc (%)'))
for name, acc in results.items():
    print('{:40s} {:12.2f}'.format(name, acc))

## Comments